In [8]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib

def load_year(year):
    X = np.load(f"wp_model1_data_x/{year}X.npy")
    y = np.load(f"wp_model1_data_y/{year}y.npy")
    return X, y

def load_many_years(years):
    X_list, y_list = [], []
    for year in years:
        X, y = load_year(year)
        X_list.append(X)
        y_list.append(y)
    return np.vstack(X_list), np.concatenate(y_list)

def evaluate_model(train_years, test_year):
    X_train, y_train = load_many_years(train_years)
    X_test, y_test = load_year(test_year)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logit", LogisticRegression(max_iter=2000))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    return {
        "train_years": train_years,
        "test_year": test_year,
        "n_train": len(y_train),
        "n_test": len(y_test),
        "accuracy": accuracy_score(y_test, y_pred),
        "log_loss": log_loss(y_test, y_prob),
        "auc": roc_auc_score(y_test, y_prob),
    }

# -----------------------------
# choose one fixed test year
# -----------------------------
test_year = 2025

experiments = [
    [2024],
    [2023, 2024],
    [2022, 2023, 2024],
]

results = []
for train_years in experiments:
    res = evaluate_model(train_years, test_year)
    results.append(res)

print(f"\nTesting all models on {test_year}\n")
for r in results:
    print(f"Train on {r['train_years']}:")
    print(f"  n_train  = {r['n_train']}")
    print(f"  n_test   = {r['n_test']}")
    print(f"  Accuracy = {r['accuracy']:.6f}")
    print(f"  Log Loss = {r['log_loss']:.6f}")
    print(f"  AUC      = {r['auc']:.6f}")
    print()


train_years = [2022, 2023, 2024]
X_train, y_train = load_many_years(train_years)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

# save it
joblib.dump(model, "wp_model_naive.pkl")


Testing all models on 2025

Train on [2024]:
  n_train  = 37797
  n_test   = 37747
  Accuracy = 0.795348
  Log Loss = 0.416278
  AUC      = 0.885639

Train on [2023, 2024]:
  n_train  = 73321
  n_test   = 37747
  Accuracy = 0.796249
  Log Loss = 0.415688
  AUC      = 0.886133

Train on [2022, 2023, 2024]:
  n_train  = 108990
  n_test   = 37747
  Accuracy = 0.796196
  Log Loss = 0.415760
  AUC      = 0.885729



['wp_model_naive.pkl']

Accuracy: 0.7856962628613081
Log Loss: 0.4287509878081088
AUC: 0.8764789969959073
